In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    KFold,
    RepeatedKFold,
    StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_val_score,
    train_test_split
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

     

In [2]:
data = pd.read_csv("dataset/wines.csv")

X = data.drop(columns=["quality"])
y = data["quality"]

In [3]:
num_cols = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density",
    "pH", "sulphates", "alcohol"
]

cat_cols = ["type"]

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first"), cat_cols)
    ]
)

In [5]:
models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "SVM": Pipeline([
        ("prep", preprocessor),
        ("model", SVC(C=1, kernel="rbf"))
    ]),

    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            random_state=42
        ))
    ])
}

In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("===== 5-Fold CV =====")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=kf, scoring="accuracy")
    print(f"{name}: Mean={scores.mean():.4f}, Std={scores.std():.4f}")

===== 5-Fold CV =====
Logistic Regression: Mean=0.5441, Std=0.0091
SVM: Mean=0.5697, Std=0.0151
Random Forest: Mean=0.5698, Std=0.0097


In [7]:
rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

print("\n===== Repeated 5-Fold CV =====")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=rkf, scoring="accuracy")
    print(f"{name}: Mean={scores.mean():.4f}, Std={scores.std():.4f}")


===== Repeated 5-Fold CV =====
Logistic Regression: Mean=0.5428, Std=0.0091
SVM: Mean=0.5712, Std=0.0120
Random Forest: Mean=0.5691, Std=0.0109


In [8]:
rskf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

print("\n===== Repeated Stratified 5-Fold CV =====")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=rskf, scoring="accuracy")
    print(f"{name}: Mean={scores.mean():.4f}, Std={scores.std():.4f}")


===== Repeated Stratified 5-Fold CV =====
Logistic Regression: Mean=0.5434, Std=0.0100
SVM: Mean=0.5716, Std=0.0108
Random Forest: Mean=0.5675, Std=0.0092


In [9]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, rf_pipeline.predict(X_train))
test_acc = accuracy_score(y_test, rf_pipeline.predict(X_test))

print("===== Overfitting Check =====")
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy : {test_acc:.4f}")

===== Overfitting Check =====
Train accuracy: 0.5805
Test accuracy : 0.5431
